# EN トレンド ピーク日予測

Pinterest Predicts 2026 で選定された監視キーワードに対して、  
**DTW スライディングウィンドウ**でいつ頃 EN ピークを迎えるかを推定する。

Pinterest Predicts はトレンドの「上昇初期」を捉えるレポートであり、  
命名年（2026）≠ ピーク年 である点に注意。  
EN ピーク推定が 2027 前後になるのは、上昇開始から 1〜2年後にピークを迎える  
典型的なトレンドサイクルと整合している。

DTW は過去の確定トレンドパターン（10本テンプレート）との波形照合によるもので、  
LOO バックテストで MAE 3.5ヶ月・±6ヶ月以内 90% の精度を確認済み。

補完手法（Chronos / Prophet / ARIMA）は **DTW 推定への信頼度スコア算出**に使用する。  
複数手法が同じ時期を指すほど信頼度が上がる設計。

**分析フロー**
1. テンプレート選定 & LOO バックテスト（DTW 精度の事前検証）
2. DTW スライディングウィンドウで EN ピーク日を推定（主手法）
3. Chronos による独立推定（補完手法①）
4. Prophet による独立推定（補完手法②）
5. ARIMA による独立推定（補完手法③）
6. 合意度をもとに信頼度スコアを算出

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')
import logging
for _name in ['cmdstanpy', 'prophet', 'stan', 'pystan']:
    logging.getLogger(_name).setLevel(logging.ERROR)
import contextlib, io

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trend_utils import load_category, get_peak, PERIODS, get_en_series, get_mean_last52, get_status
from screen_pairs import MIN_PEAK, MIN_LAST52, get_monitoring_keywords, get_best_jp_keyword
from lag_estimator import LAG_PAIRS, estimate_en_peak_dtw

# ── 定数 ──────────────────────────────────────────────────────────────────────
FORECAST_WEEKS = 104

# ── データ終端 ────────────────────────────────────────────────────────────────
DATA_END = load_category('styles_en', PERIODS[0]).index.max()

# ── 予測対象キーワードの構築 ──────────────────────────────────────────────────
EXCLUDED_KEYWORDS = {
    'crossbody bag', 'messenger bag',
    'military jacket', 'army jacket',
    'sequin outfit', 'mid rise jeans',
    'mob wife aesthetic', 'old money aesthetic',
}
result = get_monitoring_keywords(EXCLUDED_KEYWORDS)
result['状態']      = result['peak_date'].apply(lambda x: get_status(x, DATA_END))
result['jp_keyword'] = result['concept'].apply(get_best_jp_keyword)

print(f'DATA_END: {DATA_END.strftime("%Y-%m-%d")}')
print(f'予測対象: {len(result)}件')
result[['concept', 'keyword', 'mean_last52', '状態', 'jp_keyword']].reset_index(drop=True)

## Step 1: テンプレート選定 & LOO バックテスト

DTW スライディングウィンドウマッチングに使うテンプレートを選定し、  
精度を Leave-One-Out（LOO）で検証する。

- **LAG_PAIRS**（確定 4 ペア）: EN→JP ラグが実測で確定しているペアの EN 系列
- **採用テンプレート**（6 本）: EN に明確な単発ピークがあり DTW 参照として使えるキーワード

以下でそれぞれの波形を確認する。

In [ ]:
from lag_estimator import ADOPTED_TEMPLATES, ALL_TEMPLATES

def _plot_templates(templates, title, color):
    n = len(templates)
    cols = 3
    rows = (n + cols - 1) // cols
    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=[t['label'] for t in templates],
        vertical_spacing=0.15, horizontal_spacing=0.08,
    )
    for i, t in enumerate(templates):
        r, c = i // cols + 1, i % cols + 1
        s = load_category(t['cat'], t['period'])
        if t['kw'] not in s.columns:
            continue
        s = s[t['kw']]
        peak_val, peak_date = get_peak(s)
        fig.add_trace(go.Scatter(
            x=s.index, y=s.values,
            line=dict(color=color, width=2), showlegend=False,
        ), row=r, col=c)
        fig.add_trace(go.Scatter(
            x=[peak_date], y=[peak_val],
            mode='markers+text', marker=dict(color='#E15759', size=10),
            text=[f'<b>{peak_val}</b>'], textposition='top center',
            textfont=dict(size=10), showlegend=False,
        ), row=r, col=c)
    fig.update_layout(
        title=dict(text=title, font=dict(size=13)),
        height=320 * rows, template='plotly_white',
    )
    fig.update_yaxes(range=[0, 110])
    fig.show()

lag_templates = [
    {'label': p['en_kw'], 'kw': p['en_kw'], 'cat': p['en_cat'], 'period': p['en_period']}
    for p in LAG_PAIRS
]
_plot_templates(lag_templates,     'LAG_PAIRS（確定 4 ペア）',    '#4472C4')
_plot_templates(ADOPTED_TEMPLATES, '採用テンプレート（追加 6 本）', '#2CA02C')

### LOO バックテスト（DTW 精度検証）

10 本のテンプレートを 1 本ずつ抜き（Leave-One-Out）、残り 9 本で DTW マッチングを行い  
実際のピーク日をどれだけ正確に再現できるかを検証する。

**評価方法**
- 各テンプレートについて、直近 78 週（RISE_WINDOW_WEEKS）を観測窓とする
- 対象を除いた残り 9 本でマッチングし、ピーク日を推定
- 実際のピーク日との差（ヶ月）を誤差として記録

In [3]:
RISE_WINDOW_WEEKS = 78

loo_results = []
for i, held_out in enumerate(ALL_TEMPLATES):
    leave_in = [t for j, t in enumerate(ALL_TEMPLATES) if j != i]

    s_ref = load_category(held_out['cat'], held_out['period'])
    if held_out['kw'] not in s_ref.columns:
        continue
    ref_series = s_ref[held_out['kw']]
    _, actual_peak_date = get_peak(ref_series)

    simulated_end = actual_peak_date - pd.Timedelta(weeks=1)
    obs_start     = simulated_end - pd.Timedelta(weeks=RISE_WINDOW_WEEKS - 1)
    obs_series    = ref_series[(ref_series.index >= obs_start) & (ref_series.index <= simulated_end)]

    res = estimate_en_peak_dtw(obs_series, simulated_end, leave_in)
    if res is None:
        continue

    pred_date  = res['en_peak_est']
    err_months = (pred_date.year - actual_peak_date.year) * 12 + (pred_date.month - actual_peak_date.month)

    loo_results.append({
        'template':       held_out['label'],
        'actual peak':    actual_peak_date.strftime('%Y-%m'),
        'predicted peak': pred_date.strftime('%Y-%m'),
        'error (months)': err_months,
        '|error| (months)': abs(err_months),
        'nearest match':  res['best_template'],
        'DTW distance':   round(res['dtw_distance'], 3),
    })

df_loo = pd.DataFrame(loo_results)
mae = round(df_loo['|error| (months)'].mean(), 1)
med = round(df_loo['|error| (months)'].median(), 1)
w6  = round((df_loo['|error| (months)'] <= 6).mean() * 100)
print(f'MAE: {mae} months  |  Median: {med} months  |  Within ±6 months: {w6}%')
df_loo

MAE: 3.5 months  |  Median: 1.0 months  |  Within ±6 months: 90%


,template,actual peak,predicted peak,error (months),|error| (months),nearest match,DTW distance
0,Y2K fashion,2021-09,2021-09,0,0,korean fashion,1.673
1,balletcore,2023-07,2023-08,1,1,cottagecore,7.898
2,low rise jeans,2018-12,2020-10,22,22,korean fashion,5.295
3,normcore,2018-03,2018-06,3,3,korean fashion,6.389
4,dark academia,2020-07,2020-07,0,0,barrel jeans,3.951
5,biker shorts,2018-04,2018-04,0,0,dark academia,7.427
6,cottagecore,2020-04,2020-10,6,6,balletcore,3.060
7,barrel jeans,2024-02,2024-04,2,2,dark academia,3.684
8,korean fashion,2020-07,2020-07,0,0,low rise jeans,5.202
9,barbiecore,2023-07,2023-08,1,1,cottagecore,5.151


In [4]:
fig_loo = make_subplots(
    rows=1, cols=2,
    subplot_titles=['|Error| distribution (months)', 'Actual vs Predicted peak'],
    horizontal_spacing=0.12,
)

colors_bar = ['#E15759' if v > 6 else '#4472C4' for v in df_loo['|error| (months)']]
fig_loo.add_trace(go.Bar(
    x=df_loo['template'], y=df_loo['|error| (months)'],
    marker_color=colors_bar, showlegend=False,
    text=df_loo['|error| (months)'].astype(str) + 'M',
    textposition='outside',
), row=1, col=1)
fig_loo.add_hline(y=6, line=dict(color='orange', dash='dash', width=1.5), row=1, col=1)

actual_m  = df_loo['actual peak'].apply(lambda s: int(s[:4]) * 12 + int(s[5:7]))
predict_m = df_loo['predicted peak'].apply(lambda s: int(s[:4]) * 12 + int(s[5:7]))
fig_loo.add_trace(go.Scatter(
    x=actual_m, y=predict_m,
    mode='markers+text',
    marker=dict(color='#4472C4', size=10),
    text=df_loo['template'],
    textposition='top center', textfont=dict(size=8),
    showlegend=False,
), row=1, col=2)
mn, mx = min(actual_m.min(), predict_m.min()), max(actual_m.max(), predict_m.max())
fig_loo.add_trace(go.Scatter(
    x=[mn, mx], y=[mn, mx],
    mode='lines', line=dict(color='gray', dash='dash', width=1),
    showlegend=False,
), row=1, col=2)

fig_loo.update_xaxes(tickangle=30, row=1, col=1)
fig_loo.update_xaxes(title_text='Actual peak (year×12 + month)', row=1, col=2)
fig_loo.update_yaxes(title_text='|Error| (months)', row=1, col=1)
fig_loo.update_yaxes(title_text='Predicted peak (year×12 + month)', row=1, col=2)
fig_loo.update_layout(
    title=dict(text=f'LOO Backtest  (MAE={mae}M / Median={med}M / Within ±6M: {w6}%)', font=dict(size=13)),
    height=420, template='plotly_white',
)
fig_loo.show()

**考察**

9/10 のテンプレートで |誤差| ≤ 6ヶ月（中央値 1ヶ月）と高精度だが、
low rise jeans のみ 22ヶ月という外れ値になっている。

**low rise jeans が外れる理由**  
low rise jeans は 2000年代ブームのリバイバル型で、  
緩やかなプラトーを経てピークを迎える形状だった（他テンプレートはシャープな単峰型が多い）。  
DTW がピーク手前の「加速段階」でテンプレートの中盤付近にマッチしてしまい、  
残り週数を大幅に過大推定した。

**korean fashion が最近傍になりやすい理由**  
Y2K fashion・normcore・low rise jeans の LOO で最近傍として頻出している。  
これは korean fashion の波形がシャープかつ左右対称に近い単峰型で、  
「典型的な流行サイクル」に最も近い形状のため、他に近いテンプレートがないときの fallback になっている。

**精度の解釈と限界**  
外れ値の low rise jeans を除くと 9/9 が ±6ヶ月以内。  
DTW は同カテゴリ・同形状の波形には高精度だが、リバイバル型や長周期トレンドでは  
波形形状の根本的な違いから精度が落ちる可能性がある。  
テンプレート数を増やす・形状でクラスタリングして分けるといった改善余地がある。

## Step 2: DTW スライディングウィンドウによる EN ピーク日推定

各キーワードの直近の上昇波形を全 10 本のテンプレートと照合し、  
「今、テンプレートのどの位置にいるか」を DTW で特定して EN ピーク日を推定する。

- 観測窓: 直近 RISE_WINDOW_WEEKS = 78 週（約1.5年）
- 最も DTW 距離が小さいテンプレートの「ピークまでの残り週数」を EN ピーク推定に使用
- DTW 距離が小さいほど波形の一致度が高く、推定の信頼性が上がる

In [5]:
dtw_results = []
for _, row in result.iterrows():
    kw       = row['keyword']
    en_series = get_en_series(kw)
    if en_series is None:
        continue
    obs = en_series[en_series.index >= DATA_END - pd.Timedelta(weeks=RISE_WINDOW_WEEKS - 1)]
    res = estimate_en_peak_dtw(obs, DATA_END, ALL_TEMPLATES)
    if res is None:
        continue
    dtw_results.append({
        'label':           f"{row['concept']} ({kw})",
        'kw':              kw,
        'en_series':       en_series,
        'en_peak_dtw':     res['en_peak_est'],
        'weeks_remaining': res['weeks_remaining'],
        'best_template':   res['best_template'],
        'dtw_distance':    round(res['dtw_distance'], 3),
        '状態':            row['状態'],
    })

In [6]:
# DTW ピーク推定: 実績 + DATA_END + 推定ピーク縦線
n_plots = len(dtw_results)
cols = 3
rows = (n_plots + cols - 1) // cols
fig_dtw = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[r['label'] for r in dtw_results],
    vertical_spacing=0.14, horizontal_spacing=0.07,
)
for idx, r in enumerate(dtw_results):
    ri, ci = idx // cols + 1, idx % cols + 1
    s = r['en_series']
    fig_dtw.add_trace(go.Scatter(
        x=s.index, y=s.values,
        line=dict(color='#4472C4', width=1.5), showlegend=False,
    ), row=ri, col=ci)
    fig_dtw.add_vline(x=DATA_END.timestamp() * 1000,
        line=dict(color='gray', dash='dash', width=1), row=ri, col=ci)
    fig_dtw.add_vline(x=r['en_peak_dtw'].timestamp() * 1000,
        line=dict(color='#2CA02C', dash='dot', width=2), row=ri, col=ci)
    fig_dtw.add_annotation(
        x=r['en_peak_dtw'], y=s.max(),
        text=f"<b>{r['en_peak_dtw'].strftime('%Y-%m')}</b>",
        showarrow=True, arrowhead=2, font=dict(size=9, color='#2CA02C'),
        row=ri, col=ci,
    )

fig_dtw.update_layout(
    title=dict(text='DTW ピーク推定（緑点線: 推定ピーク / 灰破線: DATA_END）', font=dict(size=13)),
    height=300 * rows, template='plotly_white',
)
fig_dtw.update_yaxes(range=[0, 110])
fig_dtw.show()

**DTW 推定結果の読み方**

グラフの「推定ピーク縦線」はテンプレートの残り週数から機械的に算出したものであり、
誤差 ±3〜6ヶ月を許容して解釈する（LOO バックテストより）。

推定が同じテンプレートに集中している場合（Step 6 の "nearest match" が同一）は
テンプレートバイアスの可能性があるため注意。
推定の根拠と信頼性は Step 6 の合意度スコアで補完する。

---
## Step 3〜5: 補完手法（信頼度スコア算出用）

DTW 推定の信頼度スコアを算出するため、3つの古典的手法で独立した EN ピーク推定を取得する。  
いずれも「DTW と比較してどれだけ一致するか」が目的であり、単体での精度は問わない。

| 手法 | 特徴 | 上昇中キーワードでの傾向 |
|---|---|---|
| Chronos | ゼロショット基盤モデル（推奨予測長 64週超のため長期精度は低め） | データ末端をピークと判断しやすく ⚠️ になりがち |
| Prophet | トレンド・季節性を分解して外挿 | 上昇継続を前提とするため過剰外挿、ピーク日が遠くなりやすい |
| ARIMA(2,1,2) | 差分定常化して自己回帰 | 直近トレンドをほぼフラットに外挿、ほぼ全件 ⚠️ |

### Step 3: Chronos

In [7]:
import torch
from chronos import ChronosPipeline

pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map='cpu',
    torch_dtype=torch.bfloat16,
)
print('Chronos ロード完了')


`torch_dtype` is deprecated! Use `dtype` instead!


Chronos ロード完了


In [8]:
chronos_results = []
for _, row in result.iterrows():
    kw        = row['keyword']
    en_series = get_en_series(kw)
    if en_series is None:
        continue
    context = torch.tensor(en_series.values, dtype=torch.float32).unsqueeze(0)
    with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()), torch.no_grad():
        fc_samples = pipeline.predict(context, prediction_length=FORECAST_WEEKS, num_samples=20)[0].numpy()
    fc_low  = np.quantile(fc_samples, 0.1, axis=0).clip(0, 100)
    fc_med  = np.quantile(fc_samples, 0.5, axis=0).clip(0, 100)
    fc_high = np.quantile(fc_samples, 0.9, axis=0).clip(0, 100)
    future_dates = pd.date_range(
        start=en_series.index[-1] + pd.Timedelta(weeks=1),
        periods=FORECAST_WEEKS, freq='W',
    )
    chronos_results.append({
        'label':           f"{row['concept']} ({kw})",
        'kw':              kw,
        'en_series':       en_series,
        'future_dates':    future_dates,
        'fc_low':          fc_low,
        'fc_med':          fc_med,
        'fc_high':         fc_high,
        'en_peak_chronos': future_dates[int(np.argmax(fc_med))],
        'jp_keyword':      row.get('jp_keyword'),
        '状態':            row['状態'],
    })

In [9]:
n = len(chronos_results)
cols = 3
rows = (n + cols - 1) // cols
fig_ch = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[r['label'] for r in chronos_results],
    vertical_spacing=0.14, horizontal_spacing=0.07,
)
for idx, r in enumerate(chronos_results):
    row_i, col_i = idx // cols + 1, idx % cols + 1
    s, fd = r['en_series'], r['future_dates']
    fig_ch.add_trace(go.Scatter(x=s.index, y=s.values,
        line=dict(color='#4472C4', width=1.5), showlegend=False), row=row_i, col=col_i)
    fig_ch.add_trace(go.Scatter(
        x=list(fd) + list(fd[::-1]),
        y=list(r['fc_high']) + list(r['fc_low'][::-1]),
        fill='toself', fillcolor='rgba(231,107,243,0.15)',
        line=dict(color='rgba(0,0,0,0)'), showlegend=False,
    ), row=row_i, col=col_i)
    fig_ch.add_trace(go.Scatter(x=fd, y=r['fc_med'],
        line=dict(color='#E76BF3', width=1.5, dash='dash'), showlegend=False), row=row_i, col=col_i)
    fig_ch.add_vline(x=r['en_peak_chronos'].timestamp()*1000,
        line=dict(color='#E76BF3', dash='dot', width=2), row=row_i, col=col_i)
    fig_ch.add_vline(x=DATA_END.timestamp()*1000,
        line=dict(color='gray', dash='dash', width=1), row=row_i, col=col_i)

fig_ch.update_layout(
    title=dict(text='Chronos 予測（紫破線: 予測中央値 / 紫縦線: 推定ピーク / 灰縦線: DATA_END）', font=dict(size=13)),
    height=300 * rows, template='plotly_white',
)
fig_ch.update_yaxes(range=[0, 110])
fig_ch.show()


### Step 4: Prophet

In [10]:
import logging as _log
_cs = _log.getLogger('cmdstanpy')
_cs.handlers.clear()
_cs.addHandler(_log.NullHandler())
_cs.propagate = False

from prophet import Prophet

prophet_results = []
for _, row in result.iterrows():
    kw        = row['keyword']
    en_series = get_en_series(kw)
    if en_series is None:
        continue
    prophet_df = en_series.reset_index()
    prophet_df.columns = ['ds', 'y']
    prophet_df['ds'] = pd.to_datetime(prophet_df['ds'])
    prophet_df = prophet_df[prophet_df['y'].notna()]

    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                changepoint_prior_scale=0.3, interval_width=0.8)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        m.fit(prophet_df)

    prophet_fc = m.predict(m.make_future_dataframe(periods=FORECAST_WEEKS, freq='W'))
    for col in ['yhat', 'yhat_lower', 'yhat_upper']:
        prophet_fc[col] = prophet_fc[col].clip(0, 100)

    future_fc = prophet_fc[prophet_fc['ds'] > DATA_END]
    if future_fc.empty:
        continue
    peak_date = pd.Timestamp(future_fc.loc[future_fc['yhat'].idxmax(), 'ds'])

    prophet_results.append({
        'label':           f"{row['concept']} ({kw})",
        'kw':              kw,
        'en_series':       en_series,
        'prophet_fc':      prophet_fc,
        'en_peak_prophet': peak_date,
        '状態':            row['状態'],
    })

In [11]:
# Prophet 予測曲線可視化
n_plots = len(prophet_results)
cols = 3
rows = (n_plots + cols - 1) // cols
fig_pr = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[r['label'] for r in prophet_results],
    vertical_spacing=0.14, horizontal_spacing=0.07,
)
for idx, r in enumerate(prophet_results):
    ri, ci = idx // cols + 1, idx % cols + 1
    s  = r['en_series']
    fc = r['prophet_fc']
    future_fc = fc[fc['ds'] > DATA_END]

    fig_pr.add_trace(go.Scatter(
        x=s.index, y=s.values,
        line=dict(color='#4472C4', width=1.5), showlegend=False,
    ), row=ri, col=ci)
    fig_pr.add_trace(go.Scatter(
        x=list(future_fc['ds']) + list(future_fc['ds'][::-1]),
        y=list(future_fc['yhat_upper']) + list(future_fc['yhat_lower'][::-1]),
        fill='toself', fillcolor='rgba(255,127,14,0.12)',
        line=dict(color='rgba(0,0,0,0)'), showlegend=False,
    ), row=ri, col=ci)
    fig_pr.add_trace(go.Scatter(
        x=future_fc['ds'], y=future_fc['yhat'],
        line=dict(color='#FF7F0E', width=1.5, dash='dash'), showlegend=False,
    ), row=ri, col=ci)
    fig_pr.add_vline(x=DATA_END.timestamp()*1000,
        line=dict(color='gray', dash='dash', width=1), row=ri, col=ci)
    fig_pr.add_vline(x=r['en_peak_prophet'].timestamp()*1000,
        line=dict(color='#FF7F0E', dash='dot', width=2), row=ri, col=ci)
    fig_pr.add_annotation(
        x=r['en_peak_prophet'], y=s.max(),
        text=f"<b>{r['en_peak_prophet'].strftime('%Y-%m')}</b>",
        showarrow=True, arrowhead=2, font=dict(size=8, color='#FF7F0E'),
        row=ri, col=ci,
    )

fig_pr.update_layout(
    title=dict(text='Prophet 予測（橙破線: 予測曲線 / 橙点線: 推定ピーク / 灰線: DATA_END）', font=dict(size=13)),
    height=300 * rows, template='plotly_white',
)
fig_pr.update_yaxes(range=[0, 110])
fig_pr.show()


### Step 5: ARIMA(2,1,2)

In [12]:
from statsmodels.tsa.arima.model import ARIMA as StatsARIMA

arima_results = []
for _, row in result.iterrows():
    kw        = row['keyword']
    en_series = get_en_series(kw)
    if en_series is None:
        continue
    vals = en_series.fillna(0).values.astype(float)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            arima_fit = StatsARIMA(vals, order=(2, 1, 2)).fit()
        fc_vals = arima_fit.forecast(steps=FORECAST_WEEKS).clip(0, 100)
        ci_arr  = arima_fit.get_forecast(steps=FORECAST_WEEKS).conf_int()
        if hasattr(ci_arr, 'values'):
            ci_arr = ci_arr.values
        fc_low  = np.clip(ci_arr[:, 0], 0, 100)
        fc_high = np.clip(ci_arr[:, 1], 0, 100)
    except Exception:
        continue
    future_dates = pd.date_range(
        start=en_series.index[-1] + pd.Timedelta(weeks=1),
        periods=FORECAST_WEEKS, freq='W',
    )
    arima_results.append({
        'label':         f"{row['concept']} ({kw})",
        'kw':            kw,
        'en_series':     en_series,
        'future_dates':  future_dates,
        'fc_vals':       fc_vals,
        'fc_low':        fc_low,
        'fc_high':       fc_high,
        'en_peak_arima': future_dates[int(np.argmax(fc_vals))],
        '状態':          row['状態'],
    })

In [13]:
# ARIMA 予測曲線可視化
if not arima_results:
    print('ARIMA結果なし')
else:
    n_plots = len(arima_results)
    cols = 3
    rows = (n_plots + cols - 1) // cols
    fig_ar = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=[r['label'] for r in arima_results],
        vertical_spacing=0.14, horizontal_spacing=0.07,
    )
    for idx, r in enumerate(arima_results):
        ri, ci = idx // cols + 1, idx % cols + 1
        s, fd  = r['en_series'], r['future_dates']

        fig_ar.add_trace(go.Scatter(
            x=s.index, y=s.values,
            line=dict(color='#4472C4', width=1.5), showlegend=False,
        ), row=ri, col=ci)
        fig_ar.add_trace(go.Scatter(
            x=list(fd) + list(fd[::-1]),
            y=list(r['fc_high']) + list(r['fc_low'][::-1]),
            fill='toself', fillcolor='rgba(214,39,40,0.1)',
            line=dict(color='rgba(0,0,0,0)'), showlegend=False,
        ), row=ri, col=ci)
        fig_ar.add_trace(go.Scatter(
            x=fd, y=r['fc_vals'],
            line=dict(color='#D62728', width=1.5, dash='dash'), showlegend=False,
        ), row=ri, col=ci)
        fig_ar.add_vline(x=DATA_END.timestamp()*1000,
            line=dict(color='gray', dash='dash', width=1), row=ri, col=ci)
        fig_ar.add_vline(x=r['en_peak_arima'].timestamp()*1000,
            line=dict(color='#D62728', dash='dot', width=2), row=ri, col=ci)
        fig_ar.add_annotation(
            x=r['en_peak_arima'], y=s.max(),
            text=f"<b>{r['en_peak_arima'].strftime('%Y-%m')}</b>",
            showarrow=True, arrowhead=2, font=dict(size=8, color='#D62728'),
            row=ri, col=ci,
        )

    fig_ar.update_layout(
        title=dict(text='ARIMA 予測（赤破線: 予測曲線 / 赤点線: 推定ピーク / 灰線: DATA_END）', font=dict(size=13)),
        height=300 * rows, template='plotly_white',
    )
    fig_ar.update_yaxes(range=[0, 110])
    fig_ar.show()


## Step 6: 信頼度スコアの算出と解釈

DTW 推定に対して補完手法がどの程度一致するかで信頼度を判定し、  
各キーワードの「EN ピーク見通し」をまとめる。

**スコアリング基準**
- `⚠️` : DATA_END + 1ヶ月以内 → 「現在がピーク・ここから下落」と判断しており予測として無効
- 有効推定の中央値から ±6ヶ月以内に 2手法以上 → 🟢 高
- それ以外 → 🟡 中

**各補完手法の傾向と ⚠️ 頻発の理由**

| 手法 | 傾向 | 上昇中キーワードへの影響 |
|---|---|---|
| ARIMA(2,1,2) | 差分後の直近トレンドをほぼフラットに外挿 | ほぼ全件 ⚠️ |
| Chronos | データ末端のスコアをピークと判断しやすい | 多くが ⚠️ |
| Prophet | トレンド継続を前提に大きく外挿 | ピーク日が過剰に遠くなりがち |

ARIMA・Chronos の ⚠️ 頻発は上昇中キーワードでは**予想された結果**であり、予測の失敗ではない。  
Prophet が最も多く有効推定を出しており、DTW との比較対象として最も機能している。

**Pinterest Predicts 2026 → EN ピーク 2027 について**

全キーワードの DTW 推定ピークが 2027 前後に集まっているが、これは矛盾ではない。  
Pinterest Predicts はトレンドの「検索量が上昇し始めた段階」をレポートするものであり、  
命名年（2026）がピーク年を意味するわけではない。  
上昇初期から EN ピークまで 1〜2年かかる点は LAG_PAIRS の実績（Y2K: 上昇開始〜ピーク約2年、balletcore: 約1.5年）とも整合しており、  
むしろ **Pinterest Predicts の選定精度が高い**ことを示している可能性がある。

**EN ピーク見通しのまとめ（以下の表を参照）**

🟢 高信頼度: DTW + 1手法以上が独立して同時期を指しており、推定根拠が最も強い。  
🟡 中信頼度: DTW 推定のみが有効。データ蓄積により補完手法が有効化されれば精度が上がる。

**次のステップ（JP 波及推定への接続）**  
EN ピーク日が定まれば、確定 LAG_PAIRS から導出した平均ラグ（約 14ヶ月）を加算することで  
JP 波及タイミングの初期推定が得られる。  
JP 検索量が MIN_JP_SIGNAL（10）を超えた時点で JP 系列への Prophet 直接適用に切り替える。

In [14]:
dtw_map = {r['kw']: r['en_peak_dtw']    for r in dtw_results}
ch_map  = {r['kw']: r['en_peak_chronos'] for r in chronos_results}
pr_map  = {r['kw']: r['en_peak_prophet'] for r in prophet_results}
ar_map  = {r['kw']: r['en_peak_arima']   for r in arima_results}

# DATA_END + 1ヶ月以内は「予測放棄」とみなす
VALID_FROM       = DATA_END + pd.DateOffset(months=1)
AGREE_THRESHOLD  = 6   # 有効推定の中央値との差がこれ以下（月）を「一致」とみなす
SIGNAL_THRESHOLD = 15  # mean_last52 のシグナル判定閾値

concept_map = dict(zip(result['keyword'], result['concept']))

def is_valid(peak):
    return peak is not None and peak > VALID_FROM

comparison = []
for _, row in result.iterrows():
    kw = row['keyword']
    peaks = {
        'DTW':     dtw_map.get(kw),
        'Chronos': ch_map.get(kw),
        'Prophet': pr_map.get(kw),
        'ARIMA':   ar_map.get(kw),
    }
    valid = {k: v for k, v in peaks.items() if is_valid(v)}
    if valid:
        month_vals = sorted(v.year * 12 + v.month for v in valid.values())
        median_m   = month_vals[len(month_vals) // 2]
        n_agree    = sum(1 for m in month_vals if abs(m - median_m) <= AGREE_THRESHOLD)
        confidence = '🟢 高' if n_agree >= 2 else '🟡 中'
    else:
        n_agree, confidence = 0, '🔴 予測不能'

    rec = {
        'concept':     concept_map.get(kw, ''),
        'キーワード':   kw,
        'mean_last52': row['mean_last52'],
    }
    for method, peak in peaks.items():
        if peak is None:
            rec[method] = '-'
        elif not is_valid(peak):
            rec[method] = f'⚠️{peak.strftime("%Y-%m")}'
        else:
            rec[method] = peak.strftime('%Y-%m')
    rec['有効一致数'] = f'{n_agree}/{len(valid)}' if valid else '0/0'
    rec['信頼度']    = confidence
    rec['シグナル']  = '✅' if row['mean_last52'] >= SIGNAL_THRESHOLD else '👁'
    comparison.append(rec)

df_cmp = pd.DataFrame(comparison).sort_values('mean_last52', ascending=False).reset_index(drop=True)[
    ['concept', 'キーワード', 'mean_last52', 'DTW', 'Chronos', 'Prophet', 'ARIMA', '有効一致数', '信頼度', 'シグナル']
]
print(f'⚠️ = {VALID_FROM.strftime("%Y-%m")} 以前（DATA_END + 1ヶ月）→ 予測放棄とみなす')
df_cmp

⚠️ = 2026-03 以前（DATA_END + 1ヶ月）→ 予測放棄とみなす


,concept,キーワード,mean_last52,DTW,Chronos,Prophet,ARIMA,有効一致数,信頼度,シグナル
0,quiet luxury,quiet luxury,38.1,2027-03,2026-03,2027-05,⚠️2026-03,2/3,🟢 高,✅
1,lace outfit,lace outfit,35.9,2027-02,2026-06,2028-02,⚠️2026-03,1/3,🟡 中,✅
2,mid rise jeans,mid rise pants,29.1,2027-06,2027-02,2027-11,⚠️2026-03,3/3,🟢 高,✅
3,lace outfit,lace top outfit,7.0,2027-04,⚠️2026-03,2028-02,⚠️2026-03,1/2,🟡 中,👁
4,glamoretti fashion,layered jewelry,5.7,2026-10,2026-08,2027-11,⚠️2026-03,2/3,🟢 高,👁
